# AI Infrastructure — baseline benchmark (Colab)
Reproducible procedure. Runtime → Change runtime type → **T4 GPU** (training + GPU timing). CPU timings are the serving reference (the API runs on CPU pods).

Also run the CNN rows (`--bundle artifacts/bundle-cnn --splits data/processed-cnn/splits.npz`) to compare a 4k-parameter MLP with a 94k-parameter CNN.

**Rule: measure, do not claim.** Every number in the report must come from a CSV produced by this notebook.

In [ ]:
!git clone https://github.com/<ORG>/AdvCloud_MLOps_Proj.git project
%cd project
!pip install -q -r requirements.txt
import sys; sys.path.insert(0, "src")

## 1. Data — upload `Project.zip` (from Inside) to the Colab session

In [ ]:
!mkdir -p data/raw && unzip -o -j -q /content/Project.zip 'Project/GBG500*' -d data/raw && ls -la data/raw
!PYTHONPATH=src python -m trajclf.data

## 2. Training on GPU (fixed model version) + resource sizing
`training_time_s`, `peak_gpu_mem_mb` and hardware are printed and stored in `artifacts/bundle/meta.json`.
Use `python -m trajclf.train` instead if you want the run in MLflow.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Model 1 (champion candidate): MLP on 30 aggregate features (config.yaml)
!PYTHONPATH=src python -m trajclf.train --no-mlflow --device cuda --out artifacts/bundle
!python ai-infrastructure/sizing.py --bundle artifacts/bundle
# Model 2 (optional): 1D-CNN on the sequence (config.cnn.yaml)
!TRAJCLF_CONFIG=configs/config.cnn.yaml PYTHONPATH=src python -m trajclf.data
!TRAJCLF_CONFIG=configs/config.cnn.yaml PYTHONPATH=src python -m trajclf.train --no-mlflow --device cuda --out artifacts/bundle-cnn

## 3. Baseline: TorchScript on CPU (1 thread = 1 API pod) and on GPU

In [ ]:
!python ai-infrastructure/benchmarks/benchmark.py --bundle artifacts/bundle --runtimes torch --threads 1 --out ai-infrastructure/benchmarks/results/cpu
!python ai-infrastructure/benchmarks/benchmark.py --bundle artifacts/bundle --runtimes torch --device cuda --out ai-infrastructure/benchmarks/results/gpu

In [ ]:
import pandas as pd
cpu = pd.read_csv("ai-infrastructure/benchmarks/results/cpu/baseline-results.csv")
gpu = pd.read_csv("ai-infrastructure/benchmarks/results/gpu/baseline-results.csv")
cols = ["runtime","device","batch_size","mean_ms","p95_ms","throughput_samples_s","rss_after_load_mb","test_macro_f1"]
pd.concat([cpu, gpu])[cols].round(3)

## 4. Save
Download `ai-infrastructure/benchmarks/results/` and `ai-infrastructure/resource-sizing.md` and commit them. Keep `environment.json` with the results: it records GPU, CPU, library versions and the exact command.